> **Public release copy.** Set `<DATA_DIR>` in the CONFIG cell to the folder holding the large input files (see README / DATA availability). Internal validation cells have been removed. The small curation TSVs are included in this folder.


# 03 -- Manuscript figures (metacoder heat-trees + identity dot plots + metadata panels)

**Stage 3 of the streamlined RNAquarium virus pipeline.** Ports the figure code from old
the Figure 4 metacoder heat-trees, percent-identity dot plots, and the
metadata bar panels.

**Inputs**
- `curated_contigs.tsv`  (from 01 -- per-contig: clusterLCA, clusterLCA_curated, Category,
  rel_abundance, length, bits_NTorNR, pident_NTorNR, analysis_used, contigsbyLCAcluster,
  bioprojectsbyLCAcluster, ...)
- `curated_virus_counts_byrun_withallmetadata.tsv`  (from 02b -- counts x run + metadata, for the D panels)
- `fig4b_family_curation.tsv`  (hand-curated Category | Family | Virus for the Fig 4B table, 27 Novel+Known rows)

**Heat-trees use the metacoder taxonomy path:** `lookup_tax_data()` fetches the
taxonomy from NCBI (via taxize) keyed on `clusterLCA`, then `calc_taxon_abund()` + `heat_tree()`.
The NCBI call is wrapped in a retry loop (it can be flaky). This is intentionally
NOT the parse_tax_data/precomputed-classification shortcut used for the portal JSON in 04 -- that
shortcut is for the custom web tree, not the metacoder figures.

**Outputs (tagged by manuscript destination):**
- Figure 4A heat-tree: the published relative-abundance variant (no-label, scaled-max,
  Novel+Known subset) + a bitscore tree (kept for now). Other metric trees (contig/bioproject
  count, length) shown as a commented example -- only the node_size/node_color column changes.
- Figure 4B percent-identity dot plot (novel/known) + Category/Family/Virus TSV
- Figure 4D: 7 prevalent viruses by coarse tissue (vertical, all-gray)
- Per-tissue / devstage / year / country 53-virus bar panels (Figure 2-style)


## CONFIG -- edit paths here

In [ ]:
library(tidyverse)
library(metacoder)
library(taxize)
library(lubridate)
if (requireNamespace("pals", quietly = TRUE)) library(pals)  # for year/country palette (optional)
options(ENTREZ_KEY = Sys.getenv("ENTREZ_KEY"))   # optional: set an NCBI key to raise rate limits

# -- CONFIG -- edit paths here -----------------------------------------------
workingpath <- getwd()

curated_contigs_file  <- "curated_contigs.tsv"                              # from 01
counts_file           <- "curated_virus_counts_byrun_withallmetadata.tsv"   # from 02b

today <- Sys.Date()
cat_order <- c("Novel", "Known", "Endogenous or not fish-associated", "Insufficient Evidence")
# ----------------------------------------------------------------------------

## 1. Load curated data

In [ ]:
curated <- read_tsv(file.path(workingpath, curated_contigs_file), show_col_types = FALSE)
cat("Curated contigs:", nrow(curated), "| viruses:", n_distinct(curated$clusterLCA_curated), "\n")

## 2. Build the metacoder taxmap with `lookup_tax_data()`

This is the metacoder taxonomy pipeline -- NOT the parse_tax_data shortcut (that shortcut
is only for the portal JSON in 04). `lookup_tax_data()` fetches the NCBI taxonomy via taxize,
keyed on the `clusterLCA` column, and builds a taxmap whose `query_data` table holds the
per-contig metrics. The published figure uses the **Novel + Known subset** (the
`obj_viraltopabundance2`).

The NCBI call can be flaky, so it is wrapped in a retry loop. Set `ENTREZ_KEY` in your
environment to raise the rate limit.

After the lookup, `obj$data <- calc_taxon_abund(obj, "query_data")` summarises the per-contig
metrics up to per-taxon values. **Assigning to `obj$data` (not `obj$data$tax_abund`) is essential**
-- otherwise heat_tree finds two columns named `rel_abundance` and uses the NA-filled per-contig
one, crashing with "missing value where TRUE/FALSE needed".

In [ ]:
# Published Figure 4 tree = Novel + Known subset
tree_df <- curated %>% dplyr::filter(Category %in% c("Novel", "Known"))
cat("Tree input (Novel+Known):", nrow(tree_df), "contigs,",
    n_distinct(tree_df$clusterLCA_curated), "viruses\n")

# Trim to the columns metacoder needs; keep the metric columns so
# calc_taxon_abund() can summarise them: rel_abundance, length, bits_NTorNR,
# contigsbyLCAcluster, bioprojectsbyLCAcluster, plus the key column clusterLCA.
keep <- intersect(c("clusterLCA", "clusterLCA_curated", "Category", "bioproject",
                    "rel_abundance", "length", "bits_NTorNR", "pident_NTorNR", "analysis_used",
                    "contigsbyLCAcluster", "bioprojectsbyLCAcluster",
                    "clusterLCAcurated_contigcount", "clusterLCAcurated_bioprojectcount"),
                  names(tree_df))
tree_df <- tree_df %>% select(all_of(keep))

# --- lookup_tax_data with retry loop ---
lookup_with_retry <- function(df, column = "clusterLCA", max_retries = 3) {
  for (attempt in seq_len(max_retries)) {
    res <- tryCatch(
      lookup_tax_data(df, type = "taxon_name", column = column, ask = FALSE),
      error = function(e) { message("lookup attempt ", attempt, " failed: ", conditionMessage(e)); NULL })
    if (!is.null(res)) { message("lookup_tax_data OK on attempt ", attempt); return(res) }
    if (attempt < max_retries) Sys.sleep(8)
  }
  stop("lookup_tax_data failed after ", max_retries, " attempts")
}

obj <- lookup_with_retry(tree_df, column = "clusterLCA")

# IMPORTANT: assign to obj$data (the whole slot):
#   obj$data <- calc_taxon_abund(obj, "query_data")
# This makes the per-TAXON summary the primary data table. If you instead ADD it as
# obj$data$tax_abund, the per-contig query_data$rel_abundance (NA for internal taxa) stays in
# the object and heat_tree picks the wrong, NA-filled column -> "missing value where TRUE/FALSE
# needed" crash. (no cols= -> summarises all numeric cols: rel_abundance, length, bits_NTorNR,
# contigsbyLCAcluster, bioprojectsbyLCAcluster.)
obj$data <- calc_taxon_abund(obj, "query_data")
cat("Taxmap built:", length(obj$taxon_ids()), "taxa | per-taxon cols:",
    paste(setdiff(names(obj$data), "taxon_id"), collapse = ", "), "\n")

## 3. Heat-tree figures (Figure 4A)

The published tree (`relabundancenolabelscaledmax_novelknown`):
node size/color = `rel_abundance`, `node_label = taxon_names` (the "nolabel" in the filename is
just a label token, not `node_label = NA`), `make_node_legend = FALSE`, fixed
`node_size/color_interval = c(1e1, 1e7)` ("scaledmax"), davidson-harel layout, and crucially
**`set.seed(11)` right before** so the layout is reproducible. Plus the bitscore tree (kept for
now). The contig/bioproject-count and length trees are a one-line metric swap -- shown commented.

In [ ]:
# -> Figure 4A : the PUBLISHED tree ("relabundancenolabelscaledmax_novelknown")
# set.seed(11) gives a reproducible layout.
set.seed(11)  # makes the davidson-harel layout identical each run
abundance_tree <- heat_tree(obj,
  node_label = taxon_names,                 # pass taxon_names (the "nolabel" is just the filename token)
  node_size  = rel_abundance,
  node_color = rel_abundance,
  node_size_axis_label = "Abundance",
  node_label_size_range = c(0.005, 0.015),
  make_node_legend = FALSE,
  node_label_max = 800,
  edge_label_max = 500,
  tree_label_max = 800,
  node_color_digits = 0, node_size_digits = 0,
  edge_color_digits = 0, edge_size_digits = 0,
  repel_force = 1.5,
  node_size_interval  = c(1e1, 1e7),         # fixed scale ("scaledmax")
  node_color_interval = c(1e1, 1e7),
  layout = "davidson-harel", initial_layout = "reingold-tilford")
fn <- paste0("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancenolabelscaledmax_novelknown_Mar_", today)
ggsave(paste0(fn, ".pdf"), abundance_tree, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(paste0(fn, ".png"), abundance_tree, width = 44, height = 34, units = "in", limitsize = FALSE)
cat("Wrote published abundance heat-tree\n")

# -> Figure 4A : bitscore (kept for now; labels on)
set.seed(11)
bitscore_tree <- heat_tree(obj,
  node_label = taxon_names,
  node_size  = bits_NTorNR,
  node_color = bits_NTorNR,
  node_size_axis_label = "Bit score",
  node_label_size_range = c(0.005, 0.015),
  node_label_max = 800, edge_label_max = 500, tree_label_max = 800,
  node_color_digits = 0, node_size_digits = 0,
  edge_color_digits = 0, edge_size_digits = 0,
  repel_force = 1.5, background_color = "gray95",
  layout = "davidson-harel", initial_layout = "reingold-tilford")
fnb <- paste0("taxonomy_hits_viruses_fishspecific_metacodertree_bitscore_novelknown_", today)
ggsave(paste0(fnb, ".pdf"), bitscore_tree, width = 28, height = 25, units = "in", limitsize = FALSE)
ggsave(paste0(fnb, ".png"), bitscore_tree, width = 28, height = 25, units = "in", limitsize = FALSE)
cat("Wrote bitscore heat-tree\n")

# ---------------------------------------------------------------------------
# OPTIONAL (not in the manuscript figures) -- other metric trees. Identical heat_tree call,
# only node_size/node_color change (these columns are summarised into obj$data):
#   contigsbyLCAcluster      -> node_size_axis_label = "Transcripts by LCA cluster"
#   bioprojectsbyLCAcluster  -> node_size_axis_label = "BioProjects by LCA cluster"
#   length                   -> node_size_axis_label = "Contig length"
# e.g.
#   ct <- heat_tree(obj, node_label = taxon_names,
#     node_size = contigsbyLCAcluster, node_color = contigsbyLCAcluster,
#     node_size_axis_label = "Transcripts by LCA cluster",
#     repel_force = 1.5, background_color = "gray95",
#     layout = "davidson-harel", initial_layout = "reingold-tilford")
#   ggsave(paste0("..._metacodertree_contigsbyLCAcluster_novelknown_", today, ".pdf"),
#          ct, width = 44, height = 34, units = "in", limitsize = FALSE)
# ---------------------------------------------------------------------------

## 4. Percent-identity dot plots

`pident_NTorNR` vs virus, with `analysis_used` distinguishing NT vs NR. **Critical fix:**
the data labels NT hits as `NTclustered`; we recode `NTclustered -> NT` so BOTH
NT and NR appear (otherwise only NR shows). We make two figures:

- **Supplemental** = ALL 53 viruses, red(NR)/blue(NT) — faithful to
  `percentidentitydotplotcolor` (the supplemental PDF).
- **Figure 4B** = Novel + Known only, gray dot plot (virus names on y-axis) PLUS a curated
  **Category | Family | Virus** TSV. The published 4B places that 3-column table to the left of
  the dots in Illustrator; we generate the dots + the table data rather than an in-R patchwork
  composite (patchwork crashes on some version combos).

In [ ]:
# --- shared prep: recode NTclustered -> NT so both databases plot ---
dot <- curated %>%
  dplyr::filter(!is.na(pident_NTorNR)) %>%
  mutate(analysis_used = if_else(analysis_used == "NTclustered", "NT", analysis_used),
         Category = factor(Category, levels = cat_order))

# y-axis label with category prefix (matches the supplemental figure's labels)
prefix_for <- function(cat) dplyr::case_when(
  cat == "Endogenous or not fish-associated" ~ "endogenous_or_nonfish ",
  cat == "Insufficient Evidence"             ~ "insufficient_evidence ",
  TRUE ~ "")
dot <- dot %>% mutate(ylabel = paste0(prefix_for(as.character(Category)), clusterLCA_curated))

# order: Category, then by descending count within (reversed so Novel sits at top)
ord <- dot %>% count(Category, ylabel) %>% arrange(Category, desc(n)) %>% pull(ylabel)
dot <- dot %>% mutate(ylabel = factor(ylabel, levels = rev(ord)))

### 4a. Supplemental dot plot -- all 53 viruses, red/blue

In [ ]:
supp <- ggplot(dot,
    aes(x = pident_NTorNR, y = ylabel, color = analysis_used, shape = analysis_used)) +
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +
  scale_x_continuous(limits = c(0, 100)) +
  theme_grey(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit", y = NULL,
       color = "Database", shape = "Database") +
  theme(axis.text.y = element_text(size = 16), legend.position = "right")
# -> Supplemental (percentidentitydotplotcolor)
ggsave(paste0("taxonomy_hits_viruses_fishspecific_percentidentitydotplotcolor_", today, ".pdf"),
       supp, width = 24, height = 18, units = "in", limitsize = FALSE)
cat("Wrote supplemental dot plot (all viruses)\n")

### 4b. Figure 4B -- Novel + Known dot plot + Category/Family/Virus table TSV

Novel+Known only, gray (NR=gray70 triangle, NT=gray20 circle, alpha 0.2). We make the dot plot
with virus names on the y-axis, and separately write the curated **Category | Family | Virus**
table as a TSV. The published Figure 4B places that 3-column table to the left of the dots in
Illustrator -- which is how it was actually assembled. (We skip an in-R patchwork table: that
crashes on some patchwork/ggplot version combos inside `add_guides()`.)

In [ ]:
fam <- read_tsv(file.path(workingpath, "fig4b_family_curation.tsv"), show_col_types = FALSE)

nk <- dot %>% dplyr::filter(Category %in% c("Novel", "Known")) %>%
  mutate(clusterLCA_curated = as.character(clusterLCA_curated))

# order: Novel (by family then name) then Known (by family then name), matching the panel
nk_order <- fam %>%
  mutate(Category = factor(Category %>% str_replace("\\*$",""), levels = c("Novel","Known"))) %>%
  arrange(Category, Family, clusterLCA_curated) %>%
  pull(clusterLCA_curated)
nk_order <- nk_order[nk_order %in% nk$clusterLCA_curated]
nk <- nk %>% mutate(virus = factor(clusterLCA_curated, levels = rev(nk_order)))

fig4b <- ggplot(nk,
    aes(x = pident_NTorNR, y = virus, color = analysis_used, shape = analysis_used)) +
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray70", "NT" = "gray20")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +
  scale_x_continuous(limits = c(0, 100)) +
  theme_grey(base_size = 20) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent identity to nearest NCBI hit", y = NULL,
       color = "Database", shape = "Database") +
  theme(axis.text.y = element_text(size = 14), legend.position = "right")
# -> Figure 4B (dot plot; add the Category/Family/Virus columns in Illustrator)
ggsave(paste0("Figure4B_pident_", today, ".pdf"), fig4b,
       width = 16, height = 9, units = "in", limitsize = FALSE)
cat("Wrote Figure 4B dot plot\n")

# Category | Family | Virus table as a TSV, in the same top->bottom order as the plot,
# for placing alongside the dots in Illustrator.
write_tsv(fam %>% dplyr::filter(clusterLCA_curated %in% nk_order) %>%
            mutate(.ord = match(clusterLCA_curated, nk_order)) %>% arrange(.ord) %>% select(-.ord),
          file.path(workingpath, paste0("Figure4B_category_family_virus_", today, ".tsv")))
cat("Wrote Figure 4B Category/Family/Virus TSV\n")

## 5. Metadata bar panels (tissue / devstage / year / country)

These use the per-run counts + metadata from 02b. Build a single `virus_order` (Novel ->
Known -> Endogenous -> Insufficient, by total) reused across all panels, then facet bar plots.

In [ ]:
counts_meta <- read_tsv(file.path(workingpath, counts_file), show_col_types = FALSE)
run_col <- names(counts_meta)[1]

# 02b prefixes Endogenous/Insufficient virus columns with "endogenous_or_nonfish " /
# "insufficient_evidence ". Strip those to recover the clean curated name for joining to Category.
strip_prefix <- function(x) str_remove(x, "^(endogenous_or_nonfish |insufficient_evidence )")

# virus count columns = those whose stripped name matches a curated virus name
all_curated_names <- unique(curated$clusterLCA_curated)
virus_cols <- names(counts_meta)[strip_prefix(names(counts_meta)) %in% all_curated_names]
cat("Virus count columns found in 02b output:", length(virus_cols), "\n")

# long form with Category (join on the stripped/clean name)
virus_cat <- curated %>% distinct(clusterLCA_curated, Category)
long <- counts_meta %>%
  select(all_of(c(run_col, virus_cols)),
         any_of(c("tissue_curation_coarse","devstage_curation_coarse",
                  "earliest_date","submission.bioprojectsource.country"))) %>%
  pivot_longer(all_of(virus_cols), names_to = "virus", values_to = "count") %>%
  mutate(virus_clean = strip_prefix(virus)) %>%
  left_join(virus_cat, by = c("virus_clean" = "clusterLCA_curated")) %>%
  mutate(Category = factor(Category, levels = cat_order))

# --- ONE shared virus (panel) order, reused across ALL panels (D's virus_order):
#     by Category (Novel->Known->Endo->Insuff), then descending total count within each.
virus_order <- long %>%
  group_by(Category, virus) %>%
  summarise(virus_total = sum(count, na.rm = TRUE), .groups = "drop") %>%
  arrange(Category, desc(virus_total)) %>%
  pull(virus)
long <- long %>% mutate(virus = factor(virus, levels = virus_order))

# --- manual x-axis orderings ---
# tissue: alphabetical, then these 4 end-categories last
tissue_end <- c("Embryo Imprecise", "All anatomical structures", "Multi-system", "Undetermined")
tissue_all <- setdiff(unique(na.omit(long$tissue_curation_coarse)), tissue_end)
tissue_order <- c(sort(tissue_all), intersect(tissue_end, unique(long$tissue_curation_coarse)))
# devstage: fully manual order
devstage_order <- c("Embryo", "Larval", "Juvenile", "Adult", "Multi-stage", "Undetermined")

# --- manual fill palettes (exact hex) ---
tissue_colors <- c(
  'Adipose Tissue'="#eee8aa", 'Cancer or Tumor'="#8b4513", 'Cardiovascular System'="#ff7f50",
  'Cell Line'="#a9a9a9", 'Digestive System'="#6b8e23", 'Endocrine System'="#da70d6",
  'Hematopoietic System'="#dc143c", 'Liver and Biliary System'="#daa520",
  'Muscular System'="#9370db", 'Nervous System'="#008b8b", 'Renal System'="#4682b4",
  'Reproductive System'="#db7093", 'Respiratory System'="#6495ed", 'Sensory System'="#20b2aa",
  'Skeletal Element'="#bc8f8f", 'Surface Structure'="#deb887", 'Swim Bladder'="#66cdaa",
  'Embryo Imprecise'="#dc143c", 'All anatomical structures'="#eee8aa",
  'Multi-system'="#bdbdbd", 'Undetermined'="#e0e0e0")

devstage_colors <- c(
  'Zygote'="#a50026", 'Cleavage'="#d73027", 'Blastula'="#f46d43", 'Gastrula'="#fdae61",
  'Segmentation'="#fee090", 'Pharyngula'="#ffffbf", 'Hatching'="#e0f3f8", 'Embryo'="#d73027",
  'Larval'="#abd9e9", 'Juvenile'="#74add1", 'Adult'="#4575b4", 'Multi-stage'="#313695",
  'Undetermined'="#808080")

# year/country used pals::alphabet2(21) with Iron <-> ultraviolet swapped (D's tissue_colors_palsalphabet2)
make_pals_alphabet2 <- function() {
  if (!requireNamespace("pals", quietly = TRUE)) return(NULL)
  v <- pals::alphabet2(21)
  ip <- which(v == "#E2E2E2"); up <- which(v == "#B10DA1")
  if (length(ip)) v[ip] <- "#B10DA1"; if (length(up)) v[up] <- "#E2E2E2"
  as.character(v)
}
pals_alphabet2 <- make_pals_alphabet2()

# global virus order (by Category then total)
virus_order <- long %>% group_by(Category, virus) %>%
  summarise(total = sum(count, na.rm = TRUE), .groups = "drop") %>%
  arrange(Category, desc(total)) %>% pull(virus)
long <- long %>% mutate(virus = factor(virus, levels = virus_order))
cat("virus_order built:", length(virus_order), "viruses\n")

## 6. The four bar panels

Each: group by a metadata facet, sum counts, facet_wrap by virus. Tagged `# -> Figure 2 / supp`.

In [ ]:
# x_order: explicit x-axis ordering (NULL = as-is). fill_palette: named hex vector for
# scale_fill_manual (NULL = ggplot default fills).
panel <- function(long, group_col, xlab, title, file_tag,
                  x_order = NULL, fill_palette = NULL, legend = "right", ncol = 8, w = 27, h = 11) {
  if (!group_col %in% names(long)) { cat("skip", group_col, "(not in data)\n"); return(invisible()) }
  d <- long %>% dplyr::filter(!is.na(.data[[group_col]])) %>%
    group_by(virus, grp = .data[[group_col]]) %>%
    summarise(total = sum(count, na.rm = TRUE), .groups = "drop") %>%
    dplyr::filter(total > 0)
  if (!is.null(x_order)) d <- d %>% mutate(grp = factor(grp, levels = x_order))
  p <- ggplot(d, aes(x = grp, y = total, fill = grp)) +
    geom_col() +
    facet_wrap(~virus, scales = "free_y", ncol = ncol, drop = FALSE,
      labeller = labeller(virus = ~ str_remove(.x, "^(endogenous_or_nonfish |insufficient_evidence )"))) +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 6),
          strip.text = element_text(size = 7, face = "bold"),
          legend.position = legend, legend.title = element_blank(),
          plot.title = element_text(hjust = 0.5, size = 14, face = "bold")) +
    labs(x = xlab, y = "Total counts", title = title)
  if (!is.null(fill_palette)) {
    if (is.null(names(fill_palette))) {
      p <- p + scale_fill_manual(values = setNames(rep(fill_palette, length.out = nlevels(factor(d$grp))),
                                                    levels(factor(d$grp))))
    } else {
      p <- p + scale_fill_manual(values = fill_palette)
    }
  }
  ggsave(paste0(file_tag, "_", today, ".pdf"), p, width = w, height = h, limitsize = FALSE)
  cat("wrote", file_tag, "\n")
}

# country: top 12 by total count, then "Others"
country_tot <- long %>% dplyr::filter(!is.na(submission.bioprojectsource.country)) %>%
  group_by(submission.bioprojectsource.country) %>%
  summarise(t = sum(count, na.rm = TRUE), .groups = "drop") %>% arrange(desc(t))
top12 <- head(country_tot$submission.bioprojectsource.country, 12)
long_country <- long %>%
  mutate(country_display = if_else(submission.bioprojectsource.country %in% top12,
                                   submission.bioprojectsource.country, "Others"))
country_order <- c(top12, "Others")

# year: sorted ascending
long_year <- long %>% mutate(earliest_year = lubridate::year(earliest_date))
year_order <- sort(unique(na.omit(long_year$earliest_year)))

# -> Figure 2 / supplemental panels (shared virus_order; x-axes + fills per D)
# tissue/devstage/country keep a legend; year has none. devstage uses ggplot DEFAULT colors.
panel(long, "tissue_curation_coarse", "Tissue",
      "Virus distribution across tissue types", "virus_by_tissue_panels",
      x_order = tissue_order, fill_palette = tissue_colors, legend = "right")
panel(long, "devstage_curation_coarse", "Developmental stage",
      "Virus distribution across developmental stages", "virus_by_devstage_panels",
      x_order = devstage_order, fill_palette = NULL, legend = "right")   # ggplot default colors
panel(long_year %>% rename(.ed = earliest_date) %>% rename(earliest_date = earliest_year),
      "earliest_date", "SRA submission year",
      "Virus distribution across submission year", "virus_by_year_panels",
      x_order = year_order, fill_palette = pals_alphabet2, legend = "none")
panel(long_country, "country_display", "SRA submission country",
      "Virus distribution across submission country", "virus_by_country_panels",
      x_order = country_order, fill_palette = pals_alphabet2, legend = "right")

## 6. Figure 4D -- 7 prevalent viruses by coarse tissue (vertical, all-gray)

The published Figure 4D: a single-column stack of 7 bar panels (the most prevalent
fish-associated viruses) vs coarse tissue, bars all gray, ordered by descending total count.
 The influenza rename applies, so
"Influenza Z virus" is now "Zebrafish influenza B-like virus".

In [ ]:
virus_7subset <- c("Zebrafish jaw poxvirus",
                   "Zebrafish picornavirus 1",
                   "Danio blood picornavirus/Zebrafish picornavirus 2",
                   "Zebrafish influenza B-like virus",   # was "Influenza Z virus" in D
                   "Zebrafish rhabdo-like virus",
                   "Zebrafish gut calicivirus",
                   "Zebrafish jaw picornavirus")

d7 <- long %>%
  dplyr::filter(virus_clean %in% virus_7subset, !is.na(tissue_curation_coarse)) %>%
  group_by(virus_clean, tissue_curation_coarse) %>%
  summarise(total = sum(count, na.rm = TRUE), .groups = "drop") %>%
  # same tissue x-order as the 53-virus panels: alphabetical, then the 4 end-categories
  mutate(tissue_curation_coarse = factor(tissue_curation_coarse, levels = tissue_order))

# order panels by descending total count (as D did)
order7 <- d7 %>% group_by(virus_clean) %>% summarise(t = sum(total), .groups = "drop") %>%
  arrange(desc(t)) %>% pull(virus_clean)
d7 <- d7 %>% mutate(virus_clean = factor(virus_clean, levels = order7))

fig4d <- ggplot(d7, aes(x = tissue_curation_coarse, y = total)) +
  geom_col(width = 0.5, fill = "gray60") +
  facet_wrap(~virus_clean, scales = "free_y", ncol = 1) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
        strip.text = element_text(size = 7, face = "bold"),
        strip.background = element_rect(fill = "lightgray"),
        panel.border = element_rect(colour = "black", fill = NA, linewidth = 1),
        plot.title = element_text(hjust = 0.5, size = 16, face = "bold")) +
  labs(x = "Tissue", y = "Total counts",
       title = "Abundance of seven prevalent fish-associated viruses by tissue category")
# -> Figure 4D
ggsave(paste0("virus_by_tissue_7viruses_vertical_allgray_", today, ".pdf"),
       fig4d, width = 8.5, height = 13.5, limitsize = FALSE)
cat("Wrote Figure 4D (7 viruses by tissue). Found",
    n_distinct(d7$virus_clean), "of 7 requested viruses.\n")
missing7 <- setdiff(virus_7subset, unique(as.character(d7$virus_clean)))
if (length(missing7) > 0) { cat("MISSING from data:\n"); print(missing7) }

## 7. Notes

- Heat-trees use the metacoder `lookup_tax_data()` -> `heat_tree()` path,
  NOT the parse_tax_data shortcut (that is only used for the portal JSON).
- node_size/node_color metric columns (rel_abundance, bits_NTorNR, length, contigsbyLCAcluster,
  bioprojectsbyLCAcluster) are summarised onto taxa by `calc_taxon_abund(obj, "query_data")`,
  so the other metric trees are a one-line swap (see the commented block in section 3).
- Colors: tissue uses the manual `tissue_colors` hex palette; **devstage uses ggplot DEFAULT
  colors** (6 stages); year/country use `pals::alphabet2(21)` with the Iron<->ultraviolet swap
  (needs `pals`; falls back to ggplot defaults if absent). Legends: tissue/devstage/country show
  a legend; the year panel has none.
- Orderings match D exactly: shared virus panel order (Category, then total); tissue =
  alphabetical + 4 end-categories; devstage = manual; year = ascending; country = top-12 + Others.